# Μέρος 2: Εντοπισμός Χωρο-χρονικών Σημείων Ενδιαφέροντος και Εξαγωγή Χαρακτηριστικών σε Βίντεο Ανθρωπίνων Δράσεων

In [1]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from scipy import ndimage
import os
from part2_data import cv24_lab2_2_utils as utils

# read video
video = utils.read_video('part2_data/boxing/person02_boxing_d3_uncomp.avi', 200, 0)
video.shape

(120, 160, 200)

## 2.1 Χωρο-χρονικά Σημεία Ενδιαφέροντος

In [2]:
def HarrisDetector(video, s, sigma, tau, k=0.005, theta_corn=0.2):
    video = video.copy()

    # Get Gaussian kernel for video
    space_size = int(2*np.ceil(3*sigma)+1)
    time_size = int(2*np.ceil(3*tau)+1)
    space_kernel = cv2.getGaussianKernel(space_size, sigma).T[0]
    time_kernel = cv2.getGaussianKernel(time_size, tau).T[0]
    
    # Normalize and smoothen video
    video = video.astype(float)/video.max()
    video = ndimage.convolve1d(video, space_kernel, axis=0)
    video = ndimage.convolve1d(video, space_kernel, axis=1)
    video = ndimage.convolve1d(video, time_kernel, axis=2)

    # Calculate gradients
    Ly = ndimage.convolve1d(video, np.array([-1, 0, 1]), axis=0)
    Lx = ndimage.convolve1d(video, np.array([-1, 0, 1]), axis=1)
    Lt = ndimage.convolve1d(video, np.array([-1, 0, 1]), axis=2)

    # Get Gaussian kernel for gradient
    space_size = int(2*np.ceil(3*s*sigma)+1)
    time_size = int(2*np.ceil(3*s*tau)+1)
    space_kernel = cv2.getGaussianKernel(space_size, s*sigma).T[0]
    time_kernel = cv2.getGaussianKernel(time_size, s*tau).T[0]

    # Smoothen the gradient products
    Lxy = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Lx * Ly, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Lxt = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Lx * Lt, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Lyt = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Ly * Lt, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Lxx = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Lx * Lx, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Lyy = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Ly * Ly, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Ltt = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Lt * Lt, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)

    # Harris cornerness criterion
    trace = Lxx + Lyy + Ltt
    det = Lxx*(Lyy*Ltt - Lyt*Lyt) - Lxy*(Lxy*Ltt - Lyt*Lxt) + Lxt*(Lxy*Lyt - Lyy*Lxt)
    H = abs(det - k * trace**3)
    H = np.array(H)

    # # Apply threshold (condition Σ2)
    H[H < theta_corn*H.max()] = 0

    # Return 500 points with highest cornerness
    y, x, t = np.unravel_index(np.argsort(-1*H, axis=None), H.shape)
    scale = sigma*np.ones(x.shape)
    points = list(zip(x,y,t,scale))
    return np.array(points[:500])

# Test function
harris_points = HarrisDetector(video=video, s=2, sigma=4, tau=1.5)
# utils.show_detection(video, harris_points)

In [3]:
def GaborDetector(video, sigma, tau, theta_corn=0.02):
    video = video.copy()

    # Get Gaussian kernel for video
    space_size = int(2*np.ceil(3*sigma)+1)
    kernel = cv2.getGaussianKernel(space_size, sigma).T[0]

    # Normalize and smoothen video
    video = video.astype(float)/video.max()
    video = ndimage.convolve1d(video, kernel, axis=0)
    video = ndimage.convolve1d(video, kernel, axis=1)

    # Define Gabor filters
    t = np.linspace(-2*tau, 2*tau, int(4*tau+1))
    omega = 4/tau
    h_ev = np.cos(2*np.pi*omega*t)*np.exp(-t**2/(2*tau**2))
    h_od = np.sin(2*np.pi*omega*t)*np.exp(-t**2/(2*tau**2))

    # Normalize with L1 norm
    h_ev /= np.linalg.norm(h_ev, ord=1)
    h_od /= np.linalg.norm(h_od, ord=1)

    # Gabor cornerness criterion
    H = (ndimage.convolve1d(video, h_ev, axis=2)**2) + (ndimage.convolve1d(video, h_od, axis=2)**2)
    
    # Apply threshold (condition Σ2)
    H[H < theta_corn*H.max()] = 0

    # Return 500 points with highest cornerness
    y, x, t = np.unravel_index(np.argsort(-1*H, axis=None), H.shape)
    scale = sigma*np.ones(x.shape)
    points = list(zip(x,y,t,scale))
    return np.array(points[:500])

# Test function
gabor_points = GaborDetector(video=video, sigma=2, tau=1.5, theta_corn=0.2)
# utils.show_detection(video, gabor_points)

In [4]:
def getFeatures(video, detector):
    # Choose between Harris and Gabor detector
    # This will become useful later, on part 2.3
    if detector == 'Harris':
        points = HarrisDetector(video, s=2, sigma=4, tau=1.5)
    elif detector == 'Gabor':
        points = GaborDetector(video, sigma=2, tau=1.5, theta_corn=0.2)
    return points

## 2.2 Χωρο-χρονικοί Ιστογραφικοί Περιγραφητές

In [5]:
# Calculate the gradient of each frame
print("Calculating gradient")
video_grad_y = np.gradient(video, axis=0)
video_grad_x = np.gradient(video, axis=1)
video_grad = np.stack([video_grad_x, video_grad_y], axis=3)

# Calculate the TV-L1 optical flow
print("Calculating optical flow")
video_uint8 = video.astype(np.uint8)
oflow = cv2.DualTVL1OpticalFlow_create(nscales=1)
flow = [oflow.calc(video_uint8[:,:,frame], video_uint8[:,:,frame+1], None) for frame in range(video.shape[2]-1)]
flow = np.array(flow)
flow = np.moveaxis(flow, 0, -2)
del video_uint8

# Plot one sample frame
print("Plotting results")
# %matplotlib inline
frame = 42
y, x = np.mgrid[0:video[:,:,frame].shape[0], 0:video[:,:,frame].shape[1]]

fig=plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.imshow(video[:,:,frame], cmap='gray')
plt.title('Sample frame')
plt.axis('off')
plt.axis('equal')
plt.subplot(1,3,2)
plt.quiver(x, -y, video_grad_x[:,:,frame], video_grad_y[:,:,frame])
plt.title('Gradient of frame')
plt.axis('off')
plt.axis('equal')
plt.subplot(1,3,3)
plt.quiver(x, -y, flow[:,:,frame, 0], flow[:,:,frame, 1], scale=30)
plt.title('Optical flow of frame')
plt.axis('off')
plt.axis('equal')
# plt.show()

Calculating gradient
Calculating optical flow
Plotting results


(-7.95, 166.95, -124.95, 5.95)

In [6]:
def getDecriptors(video, points, nbins=8, descriptor="HOG-HOF"):
    # Initialize variables
    HOG = []
    HOF = []
    max_sigma = np.max(points[:,3]).astype(int)

    # Calculate the gradient of each frame
    if descriptor == 'HOG' or descriptor == 'HOG-HOF':
        video_grad_y = np.gradient(video, axis=0)
        video_grad_x = np.gradient(video, axis=1)
        # Zero pad by 2*max_sigma to account for borders
        video_grad_x = np.pad(video_grad_x, ((2*max_sigma, 2*max_sigma), (2*max_sigma, 2*max_sigma), (0,0)), 'constant')
        video_grad_y = np.pad(video_grad_y, ((2*max_sigma, 2*max_sigma), (2*max_sigma, 2*max_sigma), (0,0)), 'constant')

    # Calculate the TV-L1 optical flow
    if descriptor == 'HOF' or descriptor == 'HOG-HOF':
        video_uint8 = video.astype(np.uint8)
        oflow = cv2.DualTVL1OpticalFlow_create(nscales=1)
        flow = [oflow.calc(video_uint8[:,:,frame], video_uint8[:,:,frame+1], None) for frame in range(video.shape[2]-1)]
        flow = np.array(flow)
        flow = np.moveaxis(flow, 0, -2)
        flow_x = flow[:,:,:,0]
        flow_y = flow[:,:,:,1]
        del video_uint8, flow
        # Zero pad by 2*max_sigma to account for borders
        flow_x = np.pad(flow_x, ((2*max_sigma, 2*max_sigma), (2*max_sigma, 2*max_sigma), (0,1)), 'constant')
        flow_y = np.pad(flow_y, ((2*max_sigma, 2*max_sigma), (2*max_sigma, 2*max_sigma), (0,1)), 'constant')

    # Get descriptors for each point
    for p in range(points.shape[0]):
        x = points[p, 0].astype(int)
        y = points[p, 1].astype(int)
        t = points[p, 2].astype(int)
        sigma = points[p, 3].astype(int)

        # Get HOG descriptor
        if descriptor == 'HOG' or descriptor == 'HOG-HOF':
            Gx = video_grad_x[y:y+4*sigma, x:x+4*sigma, t] # due to the zero-pad, we add 2*sigma to the coordinates
            Gy = video_grad_y[y:y+4*sigma, x:x+4*sigma, t]
            desc = utils.orientation_histogram(Gx,Gy,nbins,np.array([2*sigma,2*sigma]))
            HOG.append(desc)

        # Get HOF descriptor
        if descriptor == 'HOF' or descriptor == 'HOG-HOF':
            Gx = flow_x[y:y+4*sigma, x:x+4*sigma, t]
            Gy = flow_y[y:y+4*sigma, x:x+4*sigma, t]
            desc = utils.orientation_histogram(Gx,Gy,nbins,np.array([2*sigma,2*sigma]))
            HOF.append(desc)

    return np.array(HOG + HOF)

# Test function
harris_desc = getDecriptors(video, harris_points, nbins=8)
gabor_desc = getDecriptors(video, gabor_points, nbins=8)
print(harris_desc.shape, gabor_desc.shape)

(1000, 512) (1000, 128)


## 2.3: Κατασκευή Bag of Visual Words και χρήση Support Vector Machines για την ταξινόμηση δράσεων

In [9]:
# Get training set filenames
with open('./part2_data/traininng_set.txt', 'r') as file:
    training_set = file.read().splitlines()

# Get training and test files and labels
train_videos = []
test_videos = []
train_labels = []
test_labels = []
dirs = ['boxing', 'running', 'walking']
for i, dir in enumerate(dirs):
    for file in os.listdir('./part2_data/'+dir):
        if file in training_set:
            train_videos.append(utils.read_video(f'./part2_data/{dir}/{file}', 200, 0))
            train_labels.append(i)
        else:
            test_videos.append(utils.read_video(f'./part2_data/{dir}/{file}', 200, 0))
            test_labels.append(i)

In [10]:
# Test Classifiers
detectors = ["Harris"]
descriptors = ["HOG"]

for detector in detectors:
    for descriptor in descriptors:
        message = f"Testing {detector} detector with {descriptor} descriptor"
        print('='*len(message))
        print(message)

        # Get train set and test set descriptors
        train_desc = []
        for video in train_videos:
            points = getFeatures(video, detector)
            desc = getDecriptors(video, points, nbins=8, descriptor=descriptor)
            train_desc.append(desc)
        test_desc = []
        for video in test_videos:
            points = getFeatures(video, detector)
            desc = getDecriptors(video, points, nbins=8, descriptor=descriptor)
            test_desc.append(desc)

    bow_train, bow_test = utils.bag_of_words(train_desc,test_desc,50)
    acc, pred = utils.svm_train_test(bow_train, train_labels, bow_test, test_labels)
    print(f"Accuracy: {acc}")

Testing Harris detector with HOG descriptor


KeyboardInterrupt: 